# Lab 02 Solution: Tool Calling Patterns

Build actual tool functions and a dispatcher that routes tool calls by name — the core mechanism inside every coding agent.

**What you'll learn:**
- Implementing file read/write/search tools
- Dispatcher pattern for tool routing
- Chaining multiple tool calls in sequence

No API key needed — pure Python standard library.

In [ ]:
import os
import shutil
import json
import fnmatch

WORKDIR = "/tmp/aidev-lab-02-02"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

## Step 1: Core Tool Functions

We define two fundamental tool functions that a coding agent needs:
- `read_file(path)` — Read and return file contents. Returns error string on failure.
- `write_file(path, content)` — Write content to a file. Creates parent directories if needed.

In [ ]:
def read_file(path):
    """Read and return file contents. Returns error string on failure."""
    try:
        with open(path, "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"ERROR: File not found: {path}"
    except Exception as e:
        return f"ERROR: {e}"


def write_file(path, content):
    """Write content to a file. Creates parent directories if needed."""
    try:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w") as f:
            f.write(content)
        return f"OK: Wrote {len(content)} chars to {path}"
    except Exception as e:
        return f"ERROR: {e}"

In [ ]:
# Demo
demo_path = os.path.join(WORKDIR, "demo.txt")
result = write_file(demo_path, "Hello from the coding agent!")
print(f"write_file demo: {result}")
result = read_file(demo_path)
print(f"read_file demo:  {result}")
result = read_file("/tmp/nonexistent_file_xyz.txt")
print(f"read_file error: {result}")

## Step 2: Dispatcher Pattern

A dispatcher maps tool names to functions, then routes calls:

```python
TOOLS = {
    "read_file":    read_file,
    "write_file":   write_file,
    "search_files": search_files,
}

def dispatch(tool_name, **kwargs):
    if tool_name not in TOOLS:
        return f"ERROR: Unknown tool: {tool_name}"
    return TOOLS[tool_name](**kwargs)
```

The LLM outputs a tool call like:
```json
{"tool": "write_file", "args": {"path": "app.py", "content": "..."}}
```
The dispatcher executes it and returns the result to the LLM.

## TODO 1: Implement search_files

In [ ]:
def search_files(pattern, directory=WORKDIR):
    """
    Search for files matching a glob pattern in directory.

    Args:
        pattern: glob pattern (e.g., "*.py", "*.txt")
        directory: root directory to search in

    Returns:
        List of matching file paths (absolute)
    """
    matches = []
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if fnmatch.fnmatch(filename, pattern):
                matches.append(os.path.join(root, filename))
    return matches

In [ ]:
# Create test files
write_file(os.path.join(WORKDIR, "app.py"), "print('hello')")
write_file(os.path.join(WORKDIR, "test_app.py"), "assert True")
write_file(os.path.join(WORKDIR, "README.md"), "# My App")
write_file(os.path.join(WORKDIR, "sub", "utils.py"), "def helper(): pass")

score1 = 0
checks_1 = []

result1 = search_files("*.py", WORKDIR)

if isinstance(result1, list):
    checks_1.append(("Returns a list", "PASS"))
    score1 += 1
else:
    checks_1.append(("Returns a list", "FAIL"))

py_files = result1 if isinstance(result1, list) else []
if len(py_files) >= 3:
    checks_1.append(("Finds *.py files", "PASS"))
    score1 += 1
else:
    checks_1.append((f"Finds *.py files (got {len(py_files)}, expected >= 3)", "FAIL"))

if any("sub" in p for p in py_files):
    checks_1.append(("Finds files in subdirectories", "PASS"))
    score1 += 1
else:
    checks_1.append(("Finds files in subdirectories", "FAIL"))

for check, status in checks_1:
    print(f"[{status}] {check}")

print(f"\nScore: {score1}/3")

## TODO 2: Implement tool_dispatcher

In [ ]:
def tool_dispatcher(tool_name, **kwargs):
    """Dispatch a tool call to the correct function."""
    tools = {
        "read_file": read_file,
        "write_file": write_file,
        "search_files": search_files,
    }
    if tool_name not in tools:
        return f"ERROR: Unknown tool: {tool_name}"
    return tools[tool_name](**kwargs)

In [ ]:
score2 = 0
checks_2 = []

r1 = tool_dispatcher("write_file", path=os.path.join(WORKDIR, "dispatched.txt"), content="via dispatcher")
if isinstance(r1, str) and "OK" in r1:
    checks_2.append(("Dispatches write_file", "PASS"))
    score2 += 1
else:
    checks_2.append((f"Dispatches write_file (got: {r1})", "FAIL"))

r2 = tool_dispatcher("read_file", path=os.path.join(WORKDIR, "dispatched.txt"))
if isinstance(r2, str) and "via dispatcher" in r2:
    checks_2.append(("Dispatches read_file", "PASS"))
    score2 += 1
else:
    checks_2.append((f"Dispatches read_file (got: {r2})", "FAIL"))

r3 = tool_dispatcher("delete_everything")
if isinstance(r3, str) and "error" in r3.lower():
    checks_2.append(("Handles unknown tool", "PASS"))
    score2 += 1
else:
    checks_2.append((f"Handles unknown tool (got: {r3})", "FAIL"))

for check, status in checks_2:
    print(f"[{status}] {check}")

print(f"\nScore: {score2}/3")

## TODO 3: Chain 3 Tool Calls

Simulate an agent performing a 3-step task:
1. Write a Python file using `tool_dispatcher`
2. Read it back using `tool_dispatcher`
3. Search for `*.py` files using `tool_dispatcher`

In [ ]:
# Step A: Write a file
step_a_result = tool_dispatcher(
    "write_file",
    path=os.path.join(WORKDIR, "agent_output.py"),
    content="# Generated by agent\nprint('done')"
)

# Step B: Read it back
step_b_result = tool_dispatcher(
    "read_file",
    path=os.path.join(WORKDIR, "agent_output.py")
)

# Step C: Search for .py files
step_c_result = tool_dispatcher(
    "search_files",
    pattern="*.py",
    directory=WORKDIR
)

In [ ]:
score3 = 0
checks_3 = []

if isinstance(step_a_result, str) and "OK" in step_a_result:
    checks_3.append(("Step A: write file", "PASS"))
    score3 += 1
else:
    checks_3.append((f"Step A: write file (got: {step_a_result})", "FAIL"))

if isinstance(step_b_result, str) and "Generated by agent" in step_b_result:
    checks_3.append(("Step B: read file back", "PASS"))
    score3 += 1
else:
    checks_3.append((f"Step B: read file back (got: {step_b_result})", "FAIL"))

if isinstance(step_c_result, list) and len(step_c_result) >= 1:
    checks_3.append(("Step C: search for .py files", "PASS"))
    score3 += 1
else:
    checks_3.append((f"Step C: search for .py files (got: {step_c_result})", "FAIL"))

for check, status in checks_3:
    print(f"[{status}] {check}")

print(f"\nScore: {score3}/3")

## Summary

In [ ]:
total = score1 + score2 + score3
max_total = 3 + 3 + 3

print(f"TODO 1: {score1}/3 search_files checks passed")
print(f"TODO 2: {score2}/3 dispatcher checks passed")
print(f"TODO 3: {score3}/3 chain steps passed")
print(f"\nTotal: {total}/{max_total}")
print(f"\nFiles generated in {WORKDIR}/")

### Key Takeaways

1. **Tool functions** wrap OS operations with error handling
2. A **dispatcher** routes tool calls by name to functions
3. **Agents chain** multiple tool calls to accomplish tasks